In [1]:
import sys
import os

sys.path.append(os.path.abspath("../Codes"))

from BCI_Library import plot_brain


2026-01-17 11:26:59.543398: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-17 11:26:59.615567: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-17 11:27:01.108477: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
! nvidia-smi

Mon Jan 19 09:43:02 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.06             Driver Version: 535.183.06   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Quadro RTX 4000                On  | 00000000:5B:00.0 Off |                  N/A |
| 30%   26C    P8               7W / 125W |      6MiB /  8192MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [1]:
import Model_builder


2026-01-19 10:28:15.813991: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-19 10:28:15.888397: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-19 10:28:17.367384: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models, losses, optimizers, metrics


class MultiTaskModel(tf.keras.Model):
    def __init__(
        self,
        input_shape,
        latent_dim=16,
        conv_filters=(16, 64, 256, 128),
        kernel_size=3,
        activation="leaky_relu",
        use_decoder=True,
        use_classifier=True,
        latent_activation =  None, # use tanh if you want to ccompare with handextracted features ???
        **kwargs
    ):
        super().__init__(**kwargs)

        self.input_shape_ = input_shape
        self.latent_dim = latent_dim
        self.conv_filters = conv_filters
        self.kernel_size = kernel_size
        self.activation = activation
        self.latent_activation = latent_activation

        # Build components
        self.encoder = self._build_encoder()
        self.decoder = self._build_decoder() if use_decoder else None
        self.classifier = self._build_classifier() if use_classifier else None

    # --------------------------------------------------
    # ENCODER
    # --------------------------------------------------
    def _build_encoder(self):
        inputs = layers.Input(shape=self.input_shape_, name="encoder_input")

        x = inputs

        for n_layer in range(len(self.conv_filters)):
            filters = self.conv_filters[n_layer]
            x = layers.Conv1D(filters, self.kernel_size, padding="same")(x)
            x = layers.BatchNormalization()(x)            
            x = layers.Activation(self.activation)(x)
            x = layers.MaxPooling1D()(x)                        # downsample by 2 n_filters times (4-> shape is 128/16 = 8)
            if n_layer < len(self.conv_filters) -1 :
                x = layers.Dropout(0.3)(x)

        x = layers.Flatten()(x) # 8 * 128 = 1024

        x = layers.Dense(self.latent_dim*2, activation=self.activation)(x)
        latent = layers.Dense(self.latent_dim, name="latent", activation=self.latent_activation)(x)

        return models.Model(inputs, latent, name="encoder")

    # --------------------------------------------------
    # DECODER
    # --------------------------------------------------
    def _build_decoder(self):
        latent_inputs = layers.Input(shape=(self.latent_dim,), name="decoder_input")

        downsample_factor = 2 ** len(self.conv_filters)
        h = self.input_shape_[0] // downsample_factor
        c = self.conv_filters[-1]
        print(h,c)
        x = layers.Dense(h * c)(latent_inputs)
        x = layers.Reshape((h, c))(x)

        for filters in reversed(self.conv_filters):
            x = layers.UpSampling1D()(x)
            x = layers.Conv1D(filters, self.kernel_size, padding="same")(x)
            x = layers.Activation(self.activation)(x)

        outputs = layers.Conv1D(
            self.input_shape_[-1],
            kernel_size=3,
            padding="same",
            activation="sigmoid",
            name="reconstruction"
        )(x)

        return models.Model(latent_inputs, outputs, name="decoder")

    # --------------------------------------------------
    # CLASSIFIER
    # --------------------------------------------------
    def _build_classifier(self):
        latent_inputs = layers.Input(shape=(self.latent_dim,), name="classifier_input")

        outputs = layers.Dense(
            1,
            activation="sigmoid",
            name="classification"
        )(latent_inputs)

        return models.Model(latent_inputs, outputs, name="classifier")

    # --------------------------------------------------
    # FORWARD PASS
    # --------------------------------------------------
    def call(self, inputs, training=False):

        latent = self.encoder(inputs, training=training)

        outputs = {}

        if self.decoder is not None:
            outputs["reconstruction"] = self.decoder(latent, training=training)

        if self.classifier is not None:
            outputs["classification"] = self.classifier(latent, training=training)

        # Return single tensor if only one head exists
        if len(outputs) == 1:
            return next(iter(outputs.values()))

        return outputs

    # --------------------------------------------------
    # COMPILE FOR DIFFERENT CASES
    # --------------------------------------------------
    def compile_cases(self, optimizer, loss_reconstruction=None, loss_classification=None,
                      loss_weights={"reconstruction": 1.0, "classification": 1.0},
                      metric_reconstruction=["mse"], metric_classification=["accuracy"]):

        # Classifier-only
        if self.decoder is None and self.classifier is not None:
            self.compile(optimizer=optimizer, loss=loss_classification, metrics=metric_classification)

        # Autoencoder-only
        elif self.decoder is not None and self.classifier is None:
            self.compile(optimizer=optimizer, loss=loss_reconstruction, metrics=metric_reconstruction)

        # Multi-output
        elif self.decoder is not None and self.classifier is not None:
            loss_fns = {"reconstruction": loss_reconstruction,
            "classification": loss_classification}
            metrics_dict= {"reconstruction": ["mse"], "classification": ["accuracy"]}

            self.compile(optimizer=optimizer, loss=loss_fns, metrics=metrics_dict, loss_weights=loss_weights)

        else:
            raise RuntimeError("Model has neither decoder nor classifier.")
        

    # --------------------------------------------------
    # DEFINE FIT CASES
    # --------------------------------------------------

    def fit_cases(self, x_train, x_val, y_train=None, y_val=None, epochs=100, batch_size=32, **kwargs):
        
        # Classifier-only
        if self.decoder is None and self.classifier is not None:
            storia = self.fit()
            return storia

        # Autoencoder-only
        elif self.decoder is not None and self.classifier is None:
            storia = self.fit()
            return storia

        # Multi-output
        elif self.decoder is not None and self.classifier is not None:
            storia = self.fit()
            return storia

        else:
            raise RuntimeError("Model has neither decoder nor classifier.")  


MultiTaskModel(input_shape=(128,1), use_decoder=False)

I0000 00:00:1768816630.403913  558922 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6821 MB memory:  -> device: 0, name: Quadro RTX 4000, pci bus id: 0000:5b:00.0, compute capability: 7.5
I0000 00:00:1768816630.404864  558922 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 6761 MB memory:  -> device: 1, name: Quadro RTX 4000, pci bus id: 0000:9e:00.0, compute capability: 7.5


<MultiTaskModel name=multi_task_model, built=False>

In [2]:
tf.__version__

'2.20.0'

In [6]:
import tensorflow as tf
print(tf.__version__)
print(tf.test.is_built_with_cuda())
print("\n\n")
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs:", tf.config.list_physical_devices("GPU"))

2.20.0
True



Built with CUDA: True
GPUs: []


In [5]:
pip show tensorflow

Name: tensorflow
Version: 2.20.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /home/silvia/Documents/GitHub/BCI_Project/.venv/lib/python3.12/site-packages
Requires: absl-py, astunparse, flatbuffers, gast, google_pasta, grpcio, h5py, keras, libclang, ml_dtypes, numpy, opt_einsum, packaging, protobuf, requests, setuptools, six, tensorboard, termcolor, typing_extensions, wrapt
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [7]:
import tensorflow as tf, os
print(os.path.join(os.path.dirname(tf.__file__), "libtensorflow_framework.so.2"))

/home/silvia/Documents/GitHub/BCI_Project/.venv/lib/python3.12/site-packages/tensorflow/libtensorflow_framework.so.2


In [8]:
import tensorflow as tf
print(tf.sysconfig.get_build_info())

OrderedDict({'cpu_compiler': 'clang 18', 'cuda_compute_capabilities': ['sm_60', 'sm_70', 'sm_80', 'sm_89', 'compute_90'], 'cuda_version': '12.5.1', 'cudnn_version': '9', 'is_cuda_build': True, 'is_rocm_build': False, 'is_tensorrt_build': False})


In [ ]:
! ls /usr/local/cuda*/lib64 | grep cudnn


/usr/local/cuda-12.1/lib64:
cmake			      libnppial_static.a
libaccinj64.so		      libnppicc.so
libaccinj64.so.12.1	      libnppicc.so.12
libaccinj64.so.12.1.105       libnppicc.so.12.1.0.40
libcheckpoint.so	      libnppicc_static.a
libcublasLt.so		      libnppidei.so
libcublasLt.so.12	      libnppidei.so.12
libcublasLt.so.12.1.3.1       libnppidei.so.12.1.0.40
libcublasLt_static.a	      libnppidei_static.a
libcublas.so		      libnppif.so
libcublas.so.12		      libnppif.so.12
libcublas.so.12.1.3.1	      libnppif.so.12.1.0.40
libcublas_static.a	      libnppif_static.a
libcudadevrt.a		      libnppig.so
libcudart.so		      libnppig.so.12
libcudart.so.12		      libnppig.so.12.1.0.40
libcudart.so.12.1.105	      libnppig_static.a
libcudart_static.a	      libnppim.so
libcufft.so		      libnppim.so.12
libcufft.so.11		      libnppim.so.12.1.0.40
libcufft.so.11.0.2.54	      libnppim_static.a
libcufft_static.a	      libnppist.so
libcufft_static_nocallback.a  libnppist.so.12
libcufftw.so		      li

In [4]:
import tensorflow as tf
print(tf.config.list_physical_devices())

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [4]:
import tensorflow as tf
print(tf.sysconfig.get_build_info())

OrderedDict({'cpu_compiler': 'clang 18', 'cuda_compute_capabilities': ['sm_60', 'sm_70', 'sm_80', 'sm_89', 'compute_90'], 'cuda_version': '12.5.1', 'cudnn_version': '9', 'is_cuda_build': True, 'is_rocm_build': False, 'is_tensorrt_build': False})


In [7]:
! nvidia-smi

Mon Jan 19 09:48:41 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.06             Driver Version: 535.183.06   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Quadro RTX 4000                On  | 00000000:5B:00.0 Off |                  N/A |
| 30%   26C    P8               6W / 125W |      8MiB /  8192MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [1]:
import tensorflow as tf, os
print(os.path.join(os.path.dirname(tf.__file__), "libtensorflow_framework.so.2"))



2026-01-19 10:52:45.387756: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-19 10:52:45.463521: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-19 10:52:47.022619: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


/home/silvia/Documents/GitHub/BCI_Project/.venv/lib/python3.12/site-packages/tensorflow/libtensorflow_framework.so.2


In [2]:
import tensorflow as tf
print(tf.sysconfig.get_build_info())


OrderedDict({'cpu_compiler': 'clang 18', 'cuda_compute_capabilities': ['sm_60', 'sm_70', 'sm_80', 'sm_89', 'compute_90'], 'cuda_version': '12.5.1', 'cudnn_version': '9', 'is_cuda_build': True, 'is_rocm_build': False, 'is_tensorrt_build': False})


In [3]:
! ls /usr/local/cuda*/lib64 | grep cudnn


In [4]:
pip list | grep nvidia

Note: you may need to restart the kernel to use updated packages.


In [7]:
! pip install --upgrade \
  nvidia-cuda-runtime-cu12 \
  nvidia-cudnn-cu12


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 16.3 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.6/648.6 MB 77.9 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 80.0 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [nvidia-cudnn-cu12]nvidia-cudnn-cu12]]


In [8]:
pip list | grep nvidia


nvidia-cublas-cu12        12.9.1.4
nvidia-cuda-runtime-cu12  12.9.79
nvidia-cudnn-cu12         9.18.0.77
Note: you may need to restart the kernel to use updated packages.


In [10]:
! export LD_LIBRARY_PATH=$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cudnn/lib:$LD_LIBRARY_PATH


In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))

2026-01-19 10:56:50.409039: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-19 10:56:50.484791: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-19 10:56:52.023517: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


# CFM network

In [8]:


import keras

from keras import layers
CFM = keras.models.Sequential([
    layers.Conv1D(32, 5, input_shape=(160, 1), activation="relu", padding="same"),
    layers.Dropout(0.5),
    layers.Conv1D(64, 4, activation="relu"),
    layers.MaxPooling1D(2),
    layers.Conv1D(128, 3, activation="relu"),
    layers.MaxPooling1D(2),
    layers.Conv1D(256, 5, activation="relu", padding="same"),
    layers.Dropout(0.5),
    layers.Conv1D(128, 3, activation="relu"),
    layers.MaxPooling1D(2),
    layers.Flatten(),
    layers.Dense(50, activation="softsign"),
    layers.Dense(1, activation="sigmoid")
])

/home/silvia/Documents/GitHub/BCI_Project/.venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
CFM.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_17 (Conv1D)              │ (None, 160, 32)        │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 160, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_18 (Conv1D)              │ (None, 157, 64)        │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_15 (MaxPooling1D) │ (None, 78, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_19 (Conv1D)              │ (None, 76, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_16 (MaxPooling1D) │ (None, 38, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_20 (Conv1D)              │ (None, 38, 256)        │       164,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 38, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_21 (Conv1D)              │ (None, 36, 128)        │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_17 (MaxPooling1D) │ (None, 18, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 50)             │       115,250 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 410,981 (1.57 MB)

 Trainable params: 410,981 (1.57 MB)

 Non-trainable params: 0 (0.00 B)